# 7.02 Code Execution 도구 — PythonREPL · PythonAstREPL · E2B · Riza

에이전트가 **임의 Python 코드를 실행**해야 할 때 쓰는 도구 4종. 로컬 REPL(빠름·위험) 과 원격 샌드박스(느림·안전) 가 양극단이다.

- `PythonREPLTool` — 로컬에서 코드 문자열을 실행, 출력은 stdout 캡처
- `PythonAstREPLTool` — `ast.parse` + 마지막 표현식 반환. **DataFrame 분석에 적합**
- `E2B Data Analysis` — 원격 컨테이너 샌드박스, matplotlib 이미지·파일 업로드 지원
- `Riza Code Interpreter` — WASM 기반 초경량 격리

## 학습 목표

- 로컬 REPL vs 샌드박스의 **보안·지연·기능** tradeoff
- `PythonAstREPLTool(locals=...)` 로 데이터프레임을 도구에 공유해 에이전트가 조회
- `PythonREPLTool` 의 **위험성** (파일 시스템 접근 가능) 과 HITL 감싸기 패턴
- E2B / Riza 의 키 기반 원격 실행 경로 안내

## 언제 쓰나

- 검색 결과를 **수치 계산·집계** 해야 할 때 (환율·단위 변환 등)
- 에이전트가 **DataFrame** 을 검사하고 집계 쿼리로 답해야 할 때 → `PythonAstREPLTool` + pandas
- 사용자 제공 코드를 **격리 환경** 에서 실행해 결과만 꺼내올 때 → E2B / Riza

## 7.02.1 환경 설정

`PythonREPLTool` / `PythonAstREPLTool` 은 `langchain-experimental` 필요. E2B · Riza 는 각각 별도 패키지 + 키.

In [ ]:
# !pip install -U langchain langchain-experimental pandas

import os
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_experimental.tools import PythonREPLTool, PythonAstREPLTool  # noqa: F401
import pandas as pd  # noqa: F401

sandbox_keys = {
    "E2B":  bool(os.environ.get("E2B_API_KEY")),
    "Riza": bool(os.environ.get("RIZA_API_KEY")),
}
print(sandbox_keys)

## 7.02.2 기본 사용 — `PythonREPLTool` vs `PythonAstREPLTool`

두 도구 모두 문자열 코드를 받아 실행한다. 차이는:
- `PythonREPLTool`: stdout 을 캡처해 반환. `print(...)` 이 필요함
- `PythonAstREPLTool`: AST 로 각 문을 평가, **마지막 표현식의 값**을 반환. 표현식이 아니면 빈 문자열

In [ ]:
from langchain_experimental.tools import PythonREPLTool, PythonAstREPLTool

repl = PythonREPLTool()
out1 = repl.invoke("x = 10; y = 32; print(x + y)")
print("[PythonREPLTool] ->", repr(out1))

ast_repl = PythonAstREPLTool()
out2 = ast_repl.invoke("sum([1, 2, 3, 4, 5])")
print("[PythonAstREPLTool] ->", repr(out2))

out3 = ast_repl.invoke("a = 1\nb = 2")
print("[PythonAstREPLTool | 대입만] ->", repr(out3))

## 7.02.3 `PythonAstREPLTool(locals=...)` — DataFrame 공유

데이터 분석 에이전트의 핵심 패턴. `locals=` 에 DataFrame 을 실어 두면 **LLM 이 생성한 코드** 가 `df.query(...)`, `df.groupby(...)` 등을 바로 실행할 수 있다.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "지역": ["서울", "부산", "대구", "서울", "부산"],
    "매출": [120, 85, 60, 140, 95],
    "분기": [1, 1, 1, 2, 2],
})

df_tool = PythonAstREPLTool(locals={"df": df})

print(df_tool.invoke("df.groupby('지역')['매출'].sum().to_dict()"))
print(df_tool.invoke("df.query('분기 == 1')['매출'].mean()"))

## 7.02.4 `create_agent` 에 연결 — 계산 능력을 갖춘 에이전트

REPL 도구를 물리면 에이전트가 **"계산해야 하는 단계"** 에서 스스로 도구를 호출한다. 아래는 단순 계산기 역할.

In [ ]:
if os.environ.get("OPENAI_API_KEY"):
    from langchain.agents import create_agent

    agent = create_agent(
        model="openai:gpt-4.1-mini",
        tools=[ast_repl],
        system_prompt=(
            "산술·통계 계산이 필요하면 python_repl_ast 도구를 사용해 Python 으로 해결한다. "
            "절대 암산하지 말 것."
        ),
    )
    res = agent.invoke({
        "messages": [{"role": "user", "content": "42 * 17 을 계산하고, 그 결과를 3으로 나눈 나머지도 알려줘."}]
    })
    print(res["messages"][-1].content)
else:
    print("OPENAI_API_KEY 없음 → 에이전트 예제 스킵. 도구는 7.02.2~3에서 이미 동작 확인됨.")

## 7.02.5 E2B · Riza — 원격 샌드박스 (key-gated)

로컬 REPL 은 빠르지만 **호스트를 그대로 노출**한다 (파일 I/O · 네트워크 · 환경변수 접근 가능). 신뢰할 수 없는 코드나 사용자 입력 기반 코드는 원격 샌드박스를 권장.

- **E2B** (`e2b-code-interpreter` — `langchain-e2b` 는 PyPI 미공개) — 컨테이너 기반, matplotlib 차트·파일 업로드 지원
- **Riza** (`rizaio` + `langchain-community.tools.riza.command.ExecPython`) — WASM 격리, 낮은 cold start

In [ ]:
if sandbox_keys["E2B"]:
    from langchain.tools import tool
    from e2b_code_interpreter import Sandbox

    @tool
    def e2b_python(code: str) -> str:
        """E2B 샌드박스에서 Python 코드를 실행하고 stdout 을 반환."""
        with Sandbox() as sbx:
            exe = sbx.run_code(code)
            return "\n".join(exe.logs.stdout) or repr(exe.text)

    print(e2b_python.invoke({"code": "print(sum(range(100)))"}))
else:
    print("E2B 키 없음 → 스킵. (E2B_API_KEY 필요 + `uv pip install e2b-code-interpreter`)")

if sandbox_keys["Riza"]:
    from langchain_community.tools.riza.command import ExecPython
    riza_tool = ExecPython()
    print(riza_tool.invoke({"code": "print(2 ** 10)"}))
else:
    print("Riza 키 없음 → 스킵. (RIZA_API_KEY 필요 + `uv pip install rizaio`)")

## 7.02.6 주요 옵션 · 비교 표

| 도구 | 격리 수준 | 지연 | 주요 용도 | 보안 주의 |
|------|-----------|------|----------|----------|
| `PythonREPLTool` | **없음** (호스트) | ms | 프로토타입·디버그·계산기 | 파일·네트워크·크레덴셜 노출 |
| `PythonAstREPLTool` | **없음** (호스트) | ms | DataFrame 분석 에이전트 | 동일 — AST 파싱이어도 실행은 호스트 |
| **E2B** Sandbox | 원격 컨테이너 | 수백 ms~수 초 | 신뢰 불가 코드 실행, 차트 렌더 | 샌드박스 내 인터넷 접근 정책 확인 |
| **Riza** | WASM 격리 | 수십 ms | 저비용 격리, 대량 실행 | 시스템 콜 차단 — 네트워크도 제한 |

### `PythonAstREPLTool` 특화 필드
- `locals`, `globals`: 도구 내부 네임스페이스. 데이터·헬퍼 함수 주입 포인트
- `sanitize_input=True` (기본): 코드 앞뒤 백틱 · `python` prefix 제거

## 7.02.7 (선택) HITL 결합 — 코드 실행 승인

로컬 REPL 은 **가장 위험한 도구 중 하나**다. 실제 배포에서는 승인 게이트를 둬야 한다. 패턴:

```python
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[ast_repl],
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "python_repl_ast": {"allowed_decisions": ["approve", "edit", "reject"]},
        },
    )],
)
```

`edit` 액션으로 모델이 쓴 코드 문자열을 **검수·교정** 한 뒤 실행. 파일 I/O 가 섞이면 `reject`. HITL resume 전체 흐름은 `02_langchain/07_hitl_and_runtime` 참고.

## 체크리스트

- [ ] **프로덕션** 에 `PythonREPLTool` 직접 노출 금지 — 최소한 HITL 또는 원격 샌드박스로 감싸기
- [ ] DataFrame 분석 에이전트는 `PythonAstREPLTool(locals={"df": df})` 패턴이 표준
- [ ] E2B 차트 렌더 결과는 `exe.results` (이미지 바이트) 로 받아 첨부 기능에 연결
- [ ] Riza 는 네트워크 차단 기본 — 외부 API 호출이 필요한 코드에는 부적합
- [ ] AST REPL 은 **마지막 문이 표현식** 이어야 값을 반환. 여러 계산을 하려면 마지막을 리터럴/튜플로

## 다음

- `08_integration/07_tools/03_sql_database.ipynb` — Python 이 아닌 **SQL** 을 실행하는 도구
- `08_integration/10_sandboxes/*` — 전용 샌드박스 프로바이더 섹션 (E2B, Modal, Replit Ghostwriter 등)

## 참고

- E2B: https://e2b.dev/docs
- Riza: https://riza.io/
- `langchain-experimental` REPL: https://python.langchain.com/docs/integrations/tools/python/